# 1. Split Dependent & Independent Data (X and y)

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# --- 1. Data Loading and Feature Engineering ---
file_name = "Data/NASA_AirQuality_Excel_Analysis.xlsx"
df = pd.read_excel(file_name)
df.columns = df.columns.str.strip() # CRITICAL: Stripping whitespace from all column names

df['Date'] = pd.to_datetime(df['Date'])
# df.set_index('Date', inplace=True)
# if 'AQI_Category' in df.columns:
#     df.drop('AQI_Category', axis=1, inplace=True)

# Feature Engineering
df['Year'] = df["Date"].dt.year
df['Month'] = df["Date"].dt.month
df['DayOfWeek'] = df["Date"].dt.dayofweek
df['WeekOfYear'] = df["Date"].dt.isocalendar().week.astype(int)
df['IsWeekend'] = (df["Date"].dt.dayofweek >= 5).astype(int)
# df['AQI_lag_1'] = df.groupby('City')['AQI'].shift(1)
# df['PM2.5_MA_7'] = df.groupby('City')['PM2.5'].transform(
    # lambda x: x.rolling(window=7, min_periods=1).mean()
# )
# df['Temp_Humidity_Interaction'] = df['Temperature'] * df['Humidity']
df.drop(["Date",'AQI_Category'] , axis = 1, inplace=True)

df.dropna(inplace=True)

In [11]:
df

,City,PM2.5,PM10,NO2,SO2,CO,O3,Temperature,Humidity,WindSpeed,Rainfall,Pressure,Visibility,UVIndex,AQI,Year,Month,DayOfWeek,WeekOfYear,IsWeekend
0,Delhi,183.223221,150.148767,30.416631,35.322312,3.653451,56.541711,41.384454,72.938082,5.068853,19.605876,984.317802,11.206696,5.824295,117.51,2023,1,6,52,1
1,Delhi,117.569736,308.298392,26.424256,14.649847,2.751447,68.501035,30.300831,84.460226,7.382326,30.551176,1022.244338,5.521425,4.979675,129.24,2023,1,0,1,0
2,Delhi,94.617090,270.841643,18.289445,28.853564,2.268141,64.305361,23.626569,46.858286,4.550337,44.669458,1044.416002,8.025713,7.863482,106.79,2023,1,1,1,0
3,Delhi,99.802806,195.189174,87.967824,18.775045,2.518564,118.411576,26.623524,67.902617,1.645972,41.317040,1010.306013,8.630952,4.770402,103.92,2023,1,2,1,0
4,Delhi,121.744886,288.455268,88.791116,33.073229,3.412706,74.453021,29.996912,72.228234,8.502253,4.159749,1026.368284,4.411329,3.136453,143.32,2023,1,3,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10955,Lucknow,145.477687,364.968726,10.510030,28.906773,3.783200,103.836376,41.434269,89.192408,6.880287,9.972087,1003.204692,4.353846,2.102715,160.60,2025,12,5,52,1
10956,Lucknow,141.120737,235.659838,16.004632,5.662404,3.560327,57.139182,32.495848,31.389347,8.338376,26.555148,1002.009959,5.944148,1.454369,118.77,2025,12,6,52,1
10957,Lucknow,144.063658,330.855464,33.618578,58.992798,0.544433,17.044477,32.359182,58.238518,0.646661,24.682195,974.340308,2.445166,5.897476,157.65,2025,12,0,1,0
10958,Lucknow,66.200109,238.782940,42.603437,21.075644,1.364162,57.113519,33.373999,75.205293,1.202595,7.621028,984.593835,8.855919,8.261993,97.13,2025,12,1,1,0


In [3]:
# Assuming "df" is the full, cleaned, and feature-engineered DataFrame
X = df.drop("AQI", axis = 1) # Independent Data (Features)
y = df["AQI"]                # Dependet Data (Target)


# 2. Train, Validation and Test Split

In [4]:
from sklearn.model_selection import train_test_split

# Split into 80% Train/Val and 20% Test
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

# Further split Train/Val into 90% Train and 10% Validation
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size = 0.1, random_state = 42)



In [5]:
X_train.shape

(7891, 19)

In [6]:
X_val.shape

(877, 19)

In [7]:
y_train.shape

(7891,)

In [8]:
y_val.shape

(877,)

In [9]:
X_test.shape

(2192, 19)

In [10]:
y_test.shape

(2192,)

Output sizes:

Total rows : 10950

Train set size : 7884

Validation set size : 876

Test set size : 2190

# 3. Preprocessing: Encodings and Scaling

In [12]:
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identify feature types
categorical_features = ["City"]
numerical_features = [col for col in X_train.columns if col not in categorical_features]

# Define the Preprocessor
preprocessor = ColumnTransformer(
    transformers = [
        ("cat", OneHotEncoder(handle_unknown = "ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ],
    remainder = "passthrough"
)

# 4. Model Pipeline

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Model pipeline
print("\n--- Full Model Pipeline Construction and Training")

# Define the model
rf_model = RandomForestRegressor(n_estimators = 100, max_depth = 10, random_state = 42, n_jobs = -1)

# Define the full pipeline: Preprocessing -> Model
model_pipeline = Pipeline(steps =[
    ("preprocessor", preprocessor),
    ("regressor", rf_model)
])
# Train the full pipeline (Preprocessing and training happen in one call)
model_pipeline.fit(X_train, y_train)
y_test_pred = model_pipeline.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print(f"Model Performance on Final Test Set (Error Fixed): RMSE = {test_rmse:.2f}, R2 Score = {test_r2:.4f}")


--- Full Model Pipeline Construction and Training
Model Performance on Final Test Set (Error Fixed): RMSE = 6.39, R2 Score = 0.9723


In [14]:
import joblib

model_filename = "aqi_prediction_rf_pipeline.joblib"
joblib.dump(model_pipeline, model_filename)

['aqi_prediction_rf_pipeline.joblib']

In [15]:
dir(model_pipeline)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__sklearn_is_fitted__',
 '__sklearn_tags__',
 '__slotnames__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_can_fit_transform',
 '_can_inverse_transform',
 '_can_transform',
 '_check_method_params',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_estimator_type',
 '_final_estimator',
 '_fit',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_for_step',
 '_get_metadata_request',
 '_get_param_names',
 '_g

In [16]:
model_pipeline.feature_names_in_

array(['City', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'Temperature',
       'Humidity', 'WindSpeed', 'Rainfall', 'Pressure', 'Visibility',
       'UVIndex', 'Year', 'Month', 'DayOfWeek', 'WeekOfYear', 'IsWeekend'],
      dtype=object)